# Mini-Project 4: Rainfall Pattern & Crop Suitability Analyser

The first analysis uses the assignment's **illustrative** monthly values. The extension then uses NASA POWER precipitation estimates for 2015-2024 (10 full years) for Kampala, Gulu, and Mbarara. POWER's monthly `PRECTOTCORR` is in mm/day; multiplying by the number of days in that calendar month gives the monthly total stored in the CSV. These are gridded MERRA-2 reanalysis estimates, not rain-gauge observations.

Crop screening bands are explicit planning heuristics: maize 80-200, beans 50-150, and coffee 60-180 mm/month. The cited FAO-56 framework considers crop stage and crop coefficients (Table 11 includes a 180-day maize-grain season for East Africa and roughly 95-110-day dry-bean examples); it does **not** prescribe these monthly rainfall bands. The specific numbers are broad analyst-selected screens, not validated crop thresholds or a substitute for a local crop calendar, soil, drainage, variety, or forecast.

Sources: [NASA POWER Monthly API](https://power.larc.nasa.gov/docs/services/api/temporal/monthly/) and Allen et al., FAO Irrigation and Drainage Paper 56, [Crop evapotranspiration](https://www.fao.org/4/X0490E/X0490E00.htm), especially Chapters 6-8.

In [ ]:
from pathlib import Path

import numpy as np
from scipy.spatial.distance import cosine as scipy_cosine_distance

from src.rainfall import (
    DEFAULT_CROP_RULES,
    MONTHS,
    Region,
    classify_crop_suitability,
    cosine_similarity,
    detect_rainy_seasons,
    farmer_advisory,
    load_power_rainfall_csv,
    plot_crop_suitability,
    plot_monthly_variability,
    plot_rainfall_lines,
    regions_from_yearly_rainfall,
    similarity_matrices,
)

DATA_FILE = Path("data/rainfall_power_2015_2024.csv")
illustrative_regions = [
    Region("Kampala", [120, 140, 180, 200, 220, 180, 90, 70, 60, 100, 110, 130]),
    Region("Gulu", [8, 25, 75, 160, 190, 145, 170, 215, 175, 150, 60, 15]),
    Region("Mbarara", [70, 85, 120, 140, 90, 25, 20, 55, 100, 125, 120, 90]),
]
yearly_observations = load_power_rainfall_csv(DATA_FILE)
power_regions = regions_from_yearly_rainfall(yearly_observations)

print("Illustrative and NASA POWER mean annual rainfall (mm/year):")
for illustrative, power in zip(illustrative_regions, power_regions):
    print(
        f"{illustrative.name:8s} illustrative={illustrative.annual_total():6.1f}  "
        f"POWER 2015-2024={power.annual_total():6.1f}"
    )

## Region summaries and pairwise comparisons

Cosine similarity measures the angle between rainfall vectors, not their absolute magnitudes. Two profiles with the same month-to-month shape can therefore have cosine similarity near 1 even if one receives several times as much rain. Pearson correlation also ignores a positive linear scale change, while Euclidean distance retains magnitude differences.

In [ ]:
print("NASA POWER climatology summary:")
for region in power_regions:
    print(
        f"{region.name:8s} total={region.annual_total():7.1f} mm  "
        f"monthly mean={region.mean():5.1f} mm  "
        f"wettest={region.wettest_month()}  driest={region.driest_month()}  "
        f"monthly CV={region.coefficient_of_variation():.2f}"
    )

def print_matrix(title, matrix, names):
    print(f"\n{title}")
    print("          " + " ".join(f"{name:>10s}" for name in names))
    for name, row in zip(names, matrix):
        print(f"{name:10s}" + " ".join(f"{value:10.3f}" for value in row))

matrices = similarity_matrices(illustrative_regions)
print_matrix("Cosine similarity", matrices.cosine, matrices.region_names)
print_matrix("Pearson correlation", matrices.pearson, matrices.region_names)
print_matrix("Euclidean distance (mm)", matrices.euclidean, matrices.region_names)

first = illustrative_regions[0].monthly_rainfall_mm
second = illustrative_regions[1].monthly_rainfall_mm
numpy_similarity = cosine_similarity(first, second)
scipy_similarity = 1 - scipy_cosine_distance(first, second)
print(f"\nNumPy cosine={numpy_similarity:.8f}; SciPy equivalent={scipy_similarity:.8f}")
assert np.isclose(numpy_similarity, scipy_similarity)

## Monthly crop screening

For each crop and each region, every month is labeled drought risk, within the screening band, or waterlogging risk. A monthly rainfall total alone cannot represent effective rainfall, evapotranspiration, runoff, soil water storage, or crop growth stage.

In [ ]:
suitability = classify_crop_suitability(illustrative_regions, DEFAULT_CROP_RULES)
for region in illustrative_regions:
    print(f"\n{region.name}")
    for crop, monthly_labels in suitability[region.name].items():
        print(f"{crop:8s}: " + " | ".join(
            f"{month}: {label}" for month, label in zip(MONTHS, monthly_labels)
        ))

line_figure, _ = plot_rainfall_lines(illustrative_regions)
heatmap_figure, _ = plot_crop_suitability(illustrative_regions, DEFAULT_CROP_RULES)

## Rainy-season detection and farmer advisory

`find_peaks` is applied to a circularly extended rainfall climatology so that a peak around December/January is not lost at an array edge. Prominent peaks count as distinct seasons only when separated by a month below 80 mm; two peaks inside one continuously wet spell are grouped. This is a transparent heuristic for the exercise. As a broad climate-zone check, central/southern Kampala and Mbarara are expected to be bimodal, while northern Gulu is expected to have one extended rainy season.

The NASA POWER results reproduce that broad pattern for this 2015-2024 window. Interannual climate variability and the chosen peak/dry-month thresholds can change the detected peaks.

In [ ]:
expected_regimes = {"Kampala": "bimodal", "Gulu": "unimodal", "Mbarara": "bimodal"}
for region in power_regions:
    seasons = detect_rainy_seasons(region)
    print(f"{region.name:8s}: {seasons.classification}, peaks={seasons.peak_months}")
    assert seasons.classification == expected_regimes[region.name]

advisory = farmer_advisory(next(region for region in power_regions if region.name == "Gulu"), DEFAULT_CROP_RULES, crop="maize")
print(f"\nFarmer advisory ({len(advisory.split())} words):\n{advisory}")
assert len(advisory.split()) <= 200

## Extension: year-to-year rainfall variability

Box plots use the ten individual NASA POWER annual monthly totals, rather than only the decade climatology, so variation among years remains visible.

In [ ]:
variability_figure, _ = plot_monthly_variability(yearly_observations)